In [1]:
!pip install tensorflow keras

In [2]:
import os
print(os.listdir('/kaggle/input'))


['datasets']


In [3]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# DISABLE XLA COMPILATION explicitly to prevent Kaggle GPU freezes
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices=false"
tf.config.optimizer.set_jit(False)

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
if os.path.exists("/kaggle/input"):
    # Kaggle Kernel path
    BASE_DIR = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"
    OUTPUT_MODEL_PATH = "/kaggle/working/efficientnetb0.keras"
    TRAIN_DIR = os.path.join(BASE_DIR, "Training")
    TEST_DIR  = os.path.join(BASE_DIR, "Testing")
elif os.path.exists("/content"):
    # Google Colab path
    BASE_DIR = "/content"
    OUTPUT_MODEL_PATH = "/content/models/efficientnetb0.keras"
    TRAIN_DIR = os.path.join(BASE_DIR, "MRI images", "Training")
    TEST_DIR  = os.path.join(BASE_DIR, "MRI images", "Testing")
else:
    # Local Windows path
    BASE_DIR = r"c:\Users\Siddharth Gupta\Desktop\main_encephlo\Encephlo"
    OUTPUT_MODEL_PATH = os.path.join(BASE_DIR, "backend", "models", "efficientnetb0.keras")
    TRAIN_DIR = os.path.join(BASE_DIR, "MRI images", "Training")
    TEST_DIR  = os.path.join(BASE_DIR, "MRI images", "Testing")

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
SEED = 42

print("=" * 60)
print("LOADING DATA FOR EFFICIENTNET-B0")
print("=" * 60)

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode='categorical',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    label_mode='categorical',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    seed=SEED
)

def preprocess(image, label):
    # EfficientNet expects unscaled inputs (it has rescaling inside), but applying preprocess_input normalizes it formally.
    image = tf.keras.applications.efficientnet.preprocess_input(image)
    return image, label

train_ds = train_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

# Model Data Augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1)
], name='data_augmentation')

print("\n" + "=" * 60)
print("BUILDING EFFICIENTNET-B0 MODEL")
print("=" * 60)

# Build the base EfficientNetB0 model
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Phase 1: Freeze base model
base_model.trainable = False

# Build complete model enforcing training=False exactly like DenseNet
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_image")
x = data_augmentation(inputs)
x = base_model(x, training=False) # CRUCIAL: Ensures BN runs in inference mode!

x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x) # Creates the 1280-D vector
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name="classifier")(x)

model = keras.Model(inputs, outputs, name="efficientnetb0_classifier")

# Compile for Phase 1
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

print("\n" + "=" * 60)
print("PHASE 1: WARMUP TOP LAYERS")
print("=" * 60)

os.makedirs(os.path.dirname(OUTPUT_MODEL_PATH), exist_ok=True)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

print("\n" + "=" * 60)
print("PHASE 2: FINE-TUNING ENTIRE MODEL")
print("=" * 60)

# Unfreeze the ENTIRE model for maximum accuracy potential
base_model.trainable = True

# Ensure BatchNormalization remains frozen across the board to prevent gradient exploding
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Re-compile with an EXTREMELY slow micro-tuning learning rate to crawl out of the 94% local minimum
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_acc', factor=0.2, patience=3, min_lr=1e-8, verbose=1),
    ModelCheckpoint(filepath=OUTPUT_MODEL_PATH, monitor='val_accuracy', save_best_only=True, verbose=1)
]

print("Training for high accuracy (>96%)...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)

print("\n" + "=" * 60)
print("EXTRACTING STRICT HEADLESS MODEL")
print("=" * 60)

# The user explicitly requires a pure headless model to avoid dynamic extraction later.
# We slice the model from input directly to the 1280-D GlobalAveragePooling layer.
gap_output = model.get_layer("global_average_pooling").output
headless_model = keras.Model(inputs=model.input, outputs=gap_output, name="efficientnet_headless_1280")

headless_model.save(OUTPUT_MODEL_PATH)

print(f"\nTraining Complete. Headless EfficientNetB0 (1280-D) saved to {OUTPUT_MODEL_PATH}")


2026-03-04 20:27:53.294396: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772656073.474330      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772656073.529994      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772656073.975965      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772656073.976004      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772656073.976007      55 computation_placer.cc:177] computation placer alr

LOADING DATA FOR EFFICIENTNET-B0
Found 5600 files belonging to 4 classes.


I0000 00:00:1772656112.262603      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 1600 files belonging to 4 classes.

BUILDING EFFICIENTNET-B0 MODEL
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "efficientnetb0_classifier"

...

175/175 ━━━━━━━━━━━━━━━━━━━━ 36s 136ms/step - accuracy: 0.5906 - loss: 0.9665 - val_accuracy: 0.6687 - val_loss: 0.7744
Epoch 2/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - accuracy: 0.8079 - loss: 0.5299 - val_accuracy: 0.6931 - val_loss: 0.7799
Epoch 3/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - accuracy: 0.8237 - loss: 0.4740 - val_accuracy: 0.7431 - val_loss: 0.6935
Epoch 4/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 62ms/step - accuracy: 0.8396 - loss: 0.4368 - val_accuracy: 0.7781 - val_loss: 0.6151
Epoch 5/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 62ms/step - accuracy: 0.8343 - loss: 0.4348 - val_accuracy: 0.7619 - val_loss: 0.6425
Epoch 6/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - accuracy: 0.8501 - loss: 0.3989 - val_accuracy: 0.7519 - val_loss: 0.6647
Epoch 7/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 62ms/step - accuracy: 0.8462 - loss: 0.3984 - val_accuracy: 0.7738 - val_loss: 0.6295
Epoch 8/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - accuracy: 0.8490 - loss: 0.4014 - val_accura

E0000 00:00:1772656272.819443      55 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/efficientnetb0_classifier_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - accuracy: 0.8598 - loss: 0.3692
Epoch 1: val_accuracy improved from -inf to 0.81375, saving model to /kaggle/working/efficientnetb0.keras


/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/callback_list.py:145: UserWarning: Learning rate reduction is conditioned on metric `val_acc` which is not available. Available metrics are: accuracy,loss,val_accuracy,val_loss,learning_rate.
  callback.on_epoch_end(epoch, logs)


175/175 ━━━━━━━━━━━━━━━━━━━━ 61s 232ms/step - accuracy: 0.8599 - loss: 0.3691 - val_accuracy: 0.8138 - val_loss: 0.5012 - learning_rate: 1.0000e-05
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - accuracy: 0.8963 - loss: 0.2935
Epoch 2: val_accuracy improved from 0.81375 to 0.82187, saving model to /kaggle/working/efficientnetb0.keras
175/175 ━━━━━━━━━━━━━━━━━━━━ 38s 216ms/step - accuracy: 0.8963 - loss: 0.2934 - val_accuracy: 0.8219 - val_loss: 0.5099 - learning_rate: 1.0000e-05
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - accuracy: 0.8983 - loss: 0.2681
Epoch 3: val_accuracy improved from 0.82187 to 0.83812, saving model to /kaggle/working/efficientnetb0.keras
175/175 ━━━━━━━━━━━━━━━━━━━━ 38s 216ms/step - accuracy: 0.8983 - loss: 0.2681 - val_accuracy: 0.8381 - val_loss: 0.4586 - learning_rate: 1.0000e-05
Epoch 4/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - accuracy: 0.9063 - loss: 0.2371
Epoch 4: val_accuracy improved from 0.83812 to 0.84000, saving model to /k